# 🧪 DL-PM25-BJ — Smoke Test End-to-End trên Kaggle (CPU < 5 phút)

Kiểm tra thông suốt 100% pipeline (`download_data.py` -> `preprocess.py` -> `training_loop.py` -> `fine_tune.py` -> `run_ablation.py` -> `exp_backup.py`) với `configs/smoke_test.yaml` (2 trạm, 1 epoch, 2 batch/epoch, 4 mô hình) trước khi chạy GPU chính thức.
- **Thư mục đầu ra cách ly:** `outputs_test/`, `mlruns_test/`, `data/preprocess_test/`, `backups_test/`
- **Google Drive Folder ID:** `1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`

## Cell 1 — Kiểm tra Môi trường Kaggle CPU & Cấu trúc `/kaggle/working/`

In [ ]:
import os
import time
from pathlib import Path
import torch

SMOKE_START = time.time()
GDRIVE_FOLDER_ID = "1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL"
KAGGLE_WORK_DIR = Path("/kaggle/working/course_project")
os.environ["GDRIVE_BACKUP_FOLDER_ID"] = GDRIVE_FOLDER_ID
print(f"[ENV] PyTorch={torch.__version__} | CUDA={torch.cuda.is_available()} | WORK_DIR={KAGGLE_WORK_DIR}")

## Cell 2 — Nạp Mã nguồn & Kiểm tra Thư mục `course_project/`

In [ ]:
import os
import shutil
import time
from pathlib import Path

t0 = time.time()
REPO_URL = "https://github.com/Duy-DS/Deep_Learning_Group.git"
BRANCH_NAME = "fix/BPM25_T0305_wp03-training-audit"
KAGGLE_WORK_DIR = Path("/kaggle/working/course_project")
KAGGLE_INPUT_CODE = Path("/kaggle/input/beijing-pm25-deeplearning/course_project")

if not KAGGLE_WORK_DIR.exists():
    if KAGGLE_INPUT_CODE.exists():
        shutil.copytree(KAGGLE_INPUT_CODE, KAGGLE_WORK_DIR)
    else:
        !git clone -b {BRANCH_NAME} {REPO_URL} /kaggle/working/repo
        if Path("/kaggle/working/repo/course_project").exists():
            shutil.move("/kaggle/working/repo/course_project", str(KAGGLE_WORK_DIR))
        else:
            shutil.move("/kaggle/working/repo", str(KAGGLE_WORK_DIR))

if os.path.exists("/kaggle/working/course_project"):
    os.chdir("/kaggle/working/course_project")
os.environ["MLFLOW_TRACKING_URI"] = str(Path.cwd() / "mlruns_test")
print(f"[CWD] {os.getcwd()}")

!grep -v '^torch' requirements.txt > /tmp/req_no_torch.txt
!pip install -q -r /tmp/req_no_torch.txt mlflow google-api-python-client google-auth google-auth-oauthlib
print(f"[TIME] Cell 2 hoàn tất sau {time.time() - t0:.2f}s")

## Cell 3 — Kiểm thử Tầng Dữ liệu (`download_data.py` & `preprocess.py`)

In [ ]:
import os
import shutil
import time
from pathlib import Path
import pandas as pd

if os.path.exists("/kaggle/working/course_project"):
    os.chdir("/kaggle/working/course_project")
t0 = time.time()

raw_dir = Path("data/raw")
if not list(raw_dir.glob("PRSA_Data_*.csv")):
    kaggle_csvs = list(Path("/kaggle/input").rglob("PRSA_Data_*.csv"))
    if kaggle_csvs:
        raw_dir.mkdir(parents=True, exist_ok=True)
        for csv_p in kaggle_csvs:
            shutil.copy2(csv_p, raw_dir / csv_p.name)
    else:
        !python scripts/download_data.py --output-dir data/raw
else:
    !python scripts/download_data.py --output-dir data/raw

!python -m src.data_pipeline.preprocess --config configs/smoke_test.yaml
for name in ["train.csv", "train_dev.csv", "dev.csv", "test.csv"]:
    fp = Path("data/preprocess_test") / name
    assert fp.exists(), f"Thiếu tệp {fp}"
    df = pd.read_csv(fp)
    assert df.isna().sum().sum() == 0, f"Phát hiện NaN trong {fp}"
    print(f"[VERIFIED] {fp}: shape={df.shape}")
print(f"[TIME] Cell 3 hoàn tất sau {time.time() - t0:.2f}s")

## Cell 4 — Kiểm thử Toàn tuyến Huấn luyện, Fine-tune & Ablation (`CPU < 5 phút`)

In [ ]:
import os
import time
from pathlib import Path

if os.path.exists("/kaggle/working/course_project"):
    os.chdir("/kaggle/working/course_project")
os.environ["MLFLOW_TRACKING_URI"] = str(Path.cwd() / "mlruns_test")

t1 = time.time()
!python training_loop.py --config configs/smoke_test.yaml
print(f"[TIME] training_loop.py: {time.time() - t1:.2f}s")

t2 = time.time()
!python fine_tune.py --config configs/smoke_test.yaml
print(f"[TIME] fine_tune.py: {time.time() - t2:.2f}s")

t3 = time.time()
!python run_ablation.py --config configs/smoke_test.yaml
print(f"[TIME] run_ablation.py: {time.time() - t3:.2f}s")

artifacts = sorted([str(p) for p in Path("outputs_test").rglob("*") if p.is_file()])
print(f"[ARTIFACTS] Tổng số tệp sinh ra trong outputs_test/: {len(artifacts)}")
for p in artifacts[:25]:
    print("  -", p)

## Cell 5 — Kiểm thử Xuất CSV MLflow, Đóng gói `.zip` & Upload Drive

In [ ]:
import os
import shutil
import time
from pathlib import Path
import pandas as pd

if os.path.exists("/kaggle/working/course_project"):
    os.chdir("/kaggle/working/course_project")
t0 = time.time()

!python -m src.utils.exp_backup --mlruns-dir mlruns_test --output-dir outputs_test --backup-dir /kaggle/working/backups_test --prefix dl_pm25_bj_SMOKE_TEST --include-checkpoints --upload-gdrive

csv_path = Path("outputs_test/mlflow_runs_export.csv")
assert csv_path.exists(), "Thiếu mlflow_runs_export.csv"
df_log = pd.read_csv(csv_path)
expected_cols = ["Start Time", "Duration", "Run ID", "Name", "Source Name", "Status"]
assert all(c in df_log.columns for c in expected_cols)
print(f"[VERIFIED CSV] {len(df_log)} MLflow runs")

latest_zip = sorted(Path("/kaggle/working/backups_test").glob("dl_pm25_bj_SMOKE_TEST_*.zip"))[-1]
shutil.copy2(latest_zip, Path("/kaggle/working") / latest_zip.name)
shutil.copy2(csv_path, Path("/kaggle/working") / csv_path.name)
print(f"[KAGGLE OUTPUT] Đã lưu {latest_zip.name} và mlflow_runs_export.csv tại /kaggle/working/")
print(f"[TIME] Cell 5: {time.time() - t0:.2f}s | TỔNG THỜI GIAN SMOKE TEST: {time.time() - SMOKE_START:.2f}s")